In [1]:
!pip install python-dotenv

In [2]:
!pip install langchain-google-genai

  Using cached websockets-16.0-cp310-cp310-win_amd64.whl.metadata (7.0 kB)
  Using cached cffi-2.0.0-cp310-cp310-win_amd64.whl.metadata (2.6 kB)
  Using cached pycparser-3.0-py3-none-any.whl.metadata (8.2 kB)
   ---------------------------------------- 0.0/750.9 kB ? eta -:--:--
   ---------------------------------------- 750.9/750.9 kB 15.8 MB/s  0:00:00
Using cached websockets-16.0-cp310-cp310-win_amd64.whl (178 kB)
   ---------------------------------------- 0.0/3.5 MB ? eta -:--:--
   ---------------------------------------- 3.5/3.5 MB 25.7 MB/s  0:00:00
Using cached cffi-2.0.0-cp310-cp310-win_amd64.whl (182 kB)
Using cached pycparser-3.0-py3-none-any.whl (48 kB)

   ---- -----------------------------------  1/10 [websockets]
   -------- -------------------------------  2/10 [pycparser]
   ------------ ---------------------------  3/10 [pyasn1]
   ---------------- -----------------------  4/10 [pyasn1-modules]
   ---------------- -----------------------  4/10 [pyasn1-modules]
   --

In [8]:
%pip install python-dotenv langchain-google-genai

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install streamlit langchain langchain-groq langchain-huggingface langchain-community faiss-cpu PyPDF2 sentence-transformers python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import getpass
from langchain_groq import ChatGroq

# 1. Securely ask for the Groq API key
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")

# 2. Initialize the LLM using Meta's Llama 3.1 model running on Groq's super-fast servers
llm = ChatGroq(
    model="llama-3.1-8b-instant", 
    temperature=0
)

# 3. Ask a test question
response = llm.invoke("What is the capital of India? Answer in one sentence.")

# 4. Print the answer
print(response.content)

The capital of India is New Delhi.


In [3]:
import PyPDF2

# 1. Define the name of your PDF file (make sure it is in the same folder!)
pdf_path = "sample.pdf"

# 2. Open the PDF file in 'read binary' mode ('rb')
with open(pdf_path, 'rb') as file:
    
    # 3. Create a PDF reader object
    pdf_reader = PyPDF2.PdfReader(file)
    
    # 4. Count the total number of pages and print it
    num_pages = len(pdf_reader.pages)
    print(f"Total Pages Found: {num_pages}")
    
    # 5. Create an empty string bucket to hold all our text
    extracted_text = ""
    
    # 6. Loop through every single page one by one
    for page_num in range(num_pages):
        # Grab the specific page
        page = pdf_reader.pages[page_num]
        # Extract the text from that page and add it to our bucket
        extracted_text += page.extract_text()
        
    # 7. Print the first 500 characters to verify it worked
    print("\n--- First 500 Characters of Extracted Text ---")
    print(extracted_text[:500])


Total Pages Found: 9

--- First 500 Characters of Extracted Text ---
Bitcoin: A Peer-to-Peer Electronic Cash System
Satoshi Nakamoto
satoshin@gmx.com
www.bitcoin.org
Abstract.  A purely peer-to-peer version of electronic cash would allow online  
payments to be sent directly from one party to another without going through a  
financial institution.  Digital signatures provide part of the solution, but the main  
benefits are lost if a trusted third party is still required to prevent double-spending.  
We propose a solution to the double-spending problem using a p


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Initialize the smart text splitter with our specific rules
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len
)

# 2. Split our massive text string into a list of smaller chunk strings
chunks = text_splitter.split_text(extracted_text)

# 3. Print the total number of chunks created
print(f"Total Chunks Created: {len(chunks)}")

# 4. Print the first 2 chunks so we can verify they look good
print("\n--- CHUNK 1 ---")
print(chunks[0])

print("\n--- CHUNK 2 ---")
print(chunks[1])


Total Chunks Created: 28

--- CHUNK 1 ---
Bitcoin: A Peer-to-Peer Electronic Cash System
Satoshi Nakamoto
satoshin@gmx.com
www.bitcoin.org
Abstract.  A purely peer-to-peer version of electronic cash would allow online  
payments to be sent directly from one party to another without going through a  
financial institution.  Digital signatures provide part of the solution, but the main  
benefits are lost if a trusted third party is still required to prevent double-spending.  
We propose a solution to the double-spending problem using a peer-to-peer network.  
The network timestamps transactions by hashing them into an ongoing chain of  
hash-based proof-of-work, forming a record that cannot be changed without redoing  
the proof-of-work.  The longest chain not only serves as proof of the sequence of  
events witnessed, but proof that it came from the largest pool of CPU power.  As  
long as a majority of CPU power is controlled by nodes that are not cooperating to

--- CHUNK 2 ---
event

In [6]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# 1. Initialize the free HuggingFace Embedding model
print("Downloading/Loading Embedding Model (takes a moment the first time)...")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 2. Convert all 28 text chunks into vectors and store them in FAISS
print("Converting chunks to math vectors and building FAISS database...")
vector_store = FAISS.from_texts(chunks, embeddings)

# 3. Define a test query
query = "what is this document about?"

# 4. Perform the mathematical similarity search
print(f"\nSearching FAISS for top 2 matches to: '{query}'\n")
top_results = vector_store.similarity_search(query, k=2)

# 5. Print the results to verify
for i, result in enumerate(top_results):
    print(f"--- MATCH {i+1} ---")
    print(result.page_content)
    print("\n")


Downloading/Loading Embedding Model (takes a moment the first time)...


C:\Users\KAPOOR\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\KAPOOR\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|███████████████████████████████████████████████████

Converting chunks to math vectors and building FAISS database...

Searching FAISS for top 2 matches to: 'what is this document about?'

--- MATCH 1 ---
http://www.hashcash.org/papers/hashcash.pdf, 2002.
[7]R.C. Merkle, "Protocols for public key cryptosystems," In Proc. 1980 Symposium on Security and 
Privacy, IEEE Computer Society, pages 122-133, April 1980.
[8]W. Feller, "An introduction to probability theory and its applications," 1957.
9


--- MATCH 2 ---
8References
[1]W. Dai, "b-money," http://www.weidai.com/bmoney.txt, 1998.
[2]H. Massias, X.S. Avila, and J.-J. Quisquater, "Design of a secure timestamping service with minimal  
trust requirements," In 20th Symposium on Information Theory in the Benelux , May 1999.
[3]S. Haber, W.S. Stornetta, "How to time-stamp a digital document," In Journal of Cryptology, vol 3, no 
2, pages 99-111, 1991.
[4]D. Bayer, S. Haber, W.S. Stornetta, "Improving the efficiency and reliability of digital time-stamping,"  
In Sequences II: Methods in Com

In [8]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1. Convert our FAISS vector store into a "Retriever" tool
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

# 2. Write the Prompt (Strict instructions for the AI)
template = """Answer the question based ONLY on the following context. 
If you don't know the answer based on the context, just say "I don't know". Do not make up facts.

Context:
{context}

Question: {question}

Answer:"""
prompt = PromptTemplate.from_template(template)

# 3. Create a helper function to format the retrieved chunks into one big string
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 4. Build the modern LCEL Chain!
print("Building the modern LCEL Q&A Chain...")
qa_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 5. Define 3 test questions to ask the document
questions = [
    "What is the double-spending problem?",
    "How does the network timestamp transactions?",
    "Who wrote this whitepaper?"
]

# 6. Loop through the questions, ask the chain, and print the results
for q in questions:
    print("\n========================================================")
    print(f"❓ QUESTION: {q}")
    
    # This single line executes the entire generation pipeline!
    answer = qa_chain.invoke(q)
    print(f"\n✅ AI ANSWER: {answer}")
    
    # We run the retriever one more time just to grab and print the sources
    source_docs = retriever.invoke(q)
    print(f"\n📚 SOURCES (First 150 chars of each chunk used):")
    for i, doc in enumerate(source_docs):
        print(f"  -> Source {i+1}: {doc.page_content[:150]}...")


Building the modern LCEL Q&A Chain...

❓ QUESTION: What is the double-spending problem?

✅ AI ANSWER: The problem of a payee verifying that one of the owners did not double-spend the coin.

📚 SOURCES (First 150 chars of each chunk used):
  -> Source 1: cooperating group of attacker nodes.
12.Transactions
We define an electronic coin as a chain of digital signatures.  Each owner transfers the coin to ...
  -> Source 2: a way to initially distribute coins into circulation, since there is no central authority to issue them.  
The steady addition of a constant of amount...
  -> Source 3: Bitcoin: A Peer-to-Peer Electronic Cash System
Satoshi Nakamoto
satoshin@gmx.com
www.bitcoin.org
Abstract.  A purely peer-to-peer version of electroni...

❓ QUESTION: How does the network timestamp transactions?

✅ AI ANSWER: The network timestamps transactions using a timestamp server. The timestamp server takes a hash of a block of items to be timestamped and widely publishes the hash. Each timestamp inc